# S5.G2-2 · G3 臂 A_secure：精度与成本

> A_secure 未在 6 小时上限内训满（141 / 150 棵），按 DR-M5-010 · D-3 不报告预注册的 A_secure − A 对比，只做**描述性**比较。最直接的参照是同棵数的明文集中式模型 L4_T（XGBoost，配置与 A 相同，只训 141 棵），回答「全程秘密分享本身有没有精度代价」。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
BYTES_PER_GIB = 1024 ** 3
D4_MIN_ABS_DIFF = 0.005          # DR-M5-010 · D-4：「可分辨」但差值小于此值时降级

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g2_ssxgb_run.yaml"
run = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
PRE_PATH = REPO_ROOT / run["preregistration"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
fz = pre["freeze"]
ad = fz["arm_details"]
SSXGB_PATH = REPO_ROOT / "modules/m5_modeling/components/g3_ssxgb.py"


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("A_secure:", pre["arms"]["A_secure"], "| 数据集:", pre["compute_caps"]["a_secure_datasets"])
print("seed:", run["seed"], "| seeds:", run["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", run["step_id"], "|", run["purpose"])
print("SS-XGB 组件:", SSXGB_PATH.relative_to(REPO_ROOT), "sha256:" + sha(SSXGB_PATH))
mismatch = [p for p, h in fz["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and not mismatch
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = run["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g2_ssxgb_run.yaml sha256:0ad22448d9fc
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
A_secure: {'who_trains': 'both', 'model': 'secretflow_ss_xgb', 'flow': 'secret_shared', 'config': 'copy_of_A'} | 数据集: ['santander']
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: f634ff3（工作区含本步未提交改动）
step: S5.G2 | 按冻结的 S6.P3 在 Santander 上训练 A_secure（SS-XGB，全程秘密分享），与 A（SGB）做配对比较并报告成本
SS-XGB 组件: modules/m5_modeling/components/g3_ssxgb.py sha256:0f55ee4a8826
冻结代码指纹核对： 一致


In [2]:
from modules.m5_modeling.components import g3_arms as A
from modules.m5_modeling.components import g3_ssxgb as SX
from modules.m6_evaluation.components import metrics as MT
from modules.m6_evaluation.components import paired_eval as PE

g1 = yaml.safe_load((REPO_ROOT / "modules/m5_modeling/configs/s5_g1_santander_run.yaml").read_text(encoding="utf-8"))
arr = np.load(PRIV / run["arrays_file"])
local = np.load(PRIV / g1["local_scores_file"])
TUNING = json.loads((PRIV / run["tuning_file"]).read_text(encoding="utf-8"))
TOTAL, CAP = TUNING["A"]["best"]["num_boost_round"], pre["compute_caps"]["max_wall_clock_hours_per_fit"]
COST = SX.cost_summary(SX.load_progress(str(PRIV / run["ckpt_dir"]), SX.LABEL_PARTY), TOTAL, CAP)
T, DONE = COST["trees"], COST["within_cap"]
y, clusters = arr["dev_eval__y"], arr["dev_eval__cust"]
S = len(run["seeds"])
SC = {"A_secure": np.load(PRIV / run["ssxgb_scores_file"]), "A": np.load(PRIV / run["sgb_scores_file"])["dev_eval"]}
SC.update({a: local[a] for a in ("L0", "L1_k10", "L4")})
X4 = {n: np.hstack([arr[n + "__xg"], arr[n + "__xh"]]) for n in ("train", "dev_eval")}
refit = A.xgb_score(A.xgb_fit(X4["train"], arr["train__y"], TUNING["A"]["best"], MB, SEED0, NJ), X4["dev_eval"])
print("集中式代理 L4 重训与 S5.G1 打分最大差：%.2e" % np.abs(refit - SC["L4"]).max())
if not DONE:     # 同棵数的明文集中式参照（XGBoost 逐棵确定，前 T 棵与全量模型相同）
    cfg_t = dict(TUNING["A"]["best"], num_boost_round=T)
    SC["L4_T"] = A.xgb_score(A.xgb_fit(X4["train"], arr["train__y"], cfg_t, MB, SEED0, NJ), X4["dev_eval"])
SC = {a: np.tile(np.asarray(v).reshape(-1), (S, 1)) for a, v in SC.items()}
n_pos, base = int(y.sum()), float(y.mean())
k = run["top_k_share"]
arms = pd.DataFrame([{"arm": a, "trees": T if a in ("A_secure", "L4_T") else None,
                      "auroc": float(np.mean([MT.auc(y, s) for s in m])),
                      "lift_at_k": float(np.mean([MT.lift_at_k(y, s, k) for s in m])),
                      "recall_at_k": float(np.mean([MT.recall_at_k(y, s, k) for s in m]))} for a, m in SC.items()])
arms["precision_at_k"] = arms["lift_at_k"] * base
print("开发评估集：%d 位客户，%d 个正例；A_secure 已训 %d / %d 棵，完成且未超限：%s" % (len(y), n_pos, T, TOTAL, DONE))
print(arms.round(run["round_auc"]).to_string(index=False))

集中式代理 L4 重训与 S5.G1 打分最大差：0.00e+00


开发评估集：20351 位客户，586 个正例；A_secure 已训 141 / 150 棵，完成且未超限：False
     arm  trees  auroc  lift_at_k  recall_at_k  precision_at_k
A_secure  141.0 0.8675     5.0856       0.5085          0.1464
       A    NaN 0.8671     5.2392       0.5239          0.1509
      L0    NaN 0.8227     3.5155       0.3515          0.1012
  L1_k10    NaN 0.8244     3.3449       0.3345          0.0963
      L4    NaN 0.8681     5.1368       0.5137          0.1479
    L4_T  141.0 0.8686     5.1709       0.5171          0.1489


**读法**：A_secure（141 棵）AUROC 0.8675，与 A（隐语 SGB，150 棵）0.8671、L4（150 棵）0.8681、L4_T（141 棵）0.8686 相差都在 0.002 以内；前 10% 名单的提升度 5.09，A 为 5.24。L4 重训与 S5.G1 的打分逐位相同，参照可复现。

In [3]:
rng = np.random.default_rng(run["seed"])
alpha, power, delta = run["alpha"], run["power"], pre["primary"]["delta_of_interest"]
if DONE:
    planned = [("secondary", ("A_secure", "A"))]
else:            # 未训满或超限：不报告预注册对比，只做描述性比较（DR-M5-010 · D-3）
    planned = [("descriptive", ("A_secure", "L4_T")), ("descriptive", ("A_secure", "A")),
               ("descriptive", ("A_secure", "L1_k10"))]
out = []
for kind, (a, b) in planned:
    res = PE.paired_bootstrap(y, SC[a], SC[b], clusters, run["n_bootstrap"], alpha, rng)
    gap = PE.detectable_gap(res, n_pos, base, alpha, power, PE.MDE_CONSERVATIVE)
    out.append({"kind": kind, "contrast": "%s − %s" % (a, b), "diff": res["diff"], "ci_low": res["ci_low"],
                "ci_high": res["ci_high"], "p_value": res["p_value"], "detectable_gap": gap,
                "verdict": PE.verdict(res, gap, delta), "estimate_correlation": res["estimate_correlation"]})
con = pd.DataFrame(out)
if DONE:         # Holm 按 S6.P3 完整次要家族 9 项：S5.G1 的 8 项加本项
    ref = pd.read_csv(REPO_ROOT / run["reference_contrasts"])
    p_all = sorted(list(ref.loc[ref["kind"] == "secondary", "p_value"]) + [con.loc[0, "p_value"]])
    m = len(pre["secondary_contrasts"])
    assert len(p_all) == m
    adj, running = [], 0.0
    for rank, p in enumerate(p_all):
        running = max(running, min(1.0, (m - rank) * p))
        adj.append(running)
    con["p_holm"] = max(a for a, p in zip(adj, p_all) if p == con.loc[0, "p_value"])    # 并列取最保守
    small = con.loc[0, "verdict"].startswith("可分辨") and abs(con.loc[0, "diff"]) < D4_MIN_ABS_DIFF
    con["note"] = "不能排除运行间波动（DR-M5-010 · D-4）" if small else ""
else:
    con["p_holm"], con["note"] = None, "描述性，非预注册对比（A_secure 未在上限内训满）"
print(con.drop(columns=["estimate_correlation"]).round(run["round_auc"]).to_string(index=False))

       kind          contrast    diff  ci_low  ci_high  p_value  detectable_gap       verdict p_holm                         note
descriptive   A_secure − L4_T -0.0012 -0.0040   0.0016    0.392          0.0064 在预声明尺度上无可分辨差异   None 描述性，非预注册对比（A_secure 未在上限内训满）
descriptive      A_secure − A  0.0004 -0.0024   0.0031    0.794          0.0063 在预声明尺度上无可分辨差异   None 描述性，非预注册对比（A_secure 未在上限内训满）
descriptive A_secure − L1_k10  0.0430  0.0326   0.0537    0.000          0.0239        可分辨的提升   None 描述性，非预注册对比（A_secure 未在上限内训满）


**读法**（描述性，非预注册，不做 Holm 校正）：A_secure − L4_T = −0.001 [−0.004, 0.002]，可检出差异 0.006——同棵数下看不出全程秘密分享的精度代价；A_secure − A = +0.000 [−0.002, 0.003]；A_secure − L1_k10 = +0.043 [0.033, 0.054]，安全档仍保留相对 L1 的增量，与 A 的主对比（+0.043）相当。开发评估集 586 个正例，低于目标 700；区间只反映测试集抽样误差，A_secure 的运行间波动（D-4）未计入。

In [4]:
sgb_t = json.loads((PRIV / run["sgb_timing_file"]).read_text(encoding="utf-8"))
prog = SX.load_progress(str(PRIV / run["ckpt_dir"]), SX.LABEL_PARTY)
peak = max(SX.cost_summary(SX.load_progress(str(PRIV / run["ckpt_dir"]), p), TOTAL, CAP)["peak_rss_mb"]
           for p in SX.PARTIES)
cost = pd.DataFrame([
    {"arm": "A", "mode": "单进程模拟（debug_mode，无网络）", "trees": TOTAL, "fit_seconds": sgb_t["fit_seconds"],
     "seconds_per_tree": sgb_t["fit_seconds"] / TOTAL, "hours_for_full": sgb_t["fit_seconds"] / SX.SECONDS_PER_HOUR,
     "lo_gib_per_tree": None, "peak_rss_mb": None, "resume_overhead_seconds": None},
    {"arm": "A_secure", "mode": "两进程生产模式，本机回环", "trees": T, "fit_seconds": COST["fit_seconds"],
     "seconds_per_tree": COST["mean_tree_seconds"],
     "hours_for_full": (prog["setup_seconds"] + COST["mean_tree_seconds"] * TOTAL) / SX.SECONDS_PER_HOUR,
     "lo_gib_per_tree": COST["lo_tx_bytes"] / T / BYTES_PER_GIB, "peak_rss_mb": peak,
     "resume_overhead_seconds": COST["resume_overhead_seconds"]}])
print(cost.round(run["round_rate"]).to_string(index=False))
print("每棵树耗时比 A_secure / A：%.1f 倍（运行方式不同，只作同机相对量级）"
      % (cost.loc[1, "seconds_per_tree"] / cost.loc[0, "seconds_per_tree"]))
res_dir = REPO_ROOT / run["results_dir"]
res_dir.mkdir(parents=True, exist_ok=True)
arms.to_csv(res_dir / "arms_dev_eval.csv", index=False)
con.to_csv(res_dir / "contrasts_dev_eval.csv", index=False)
cost.to_csv(res_dir / "cost_summary.csv", index=False)
pd.DataFrame([{"n_customers": len(y), "n_positives": n_pos, "trees_planned": TOTAL, "trees_reached": T,
               "cap_hours": CAP, "within_cap": DONE}]).to_csv(res_dir / "run_summary.csv", index=False)
print("已写出公开汇总：", sorted(p.name for p in res_dir.iterdir()))

     arm                  mode  trees  fit_seconds  seconds_per_tree  hours_for_full  lo_gib_per_tree  peak_rss_mb  resume_overhead_seconds
       A 单进程模拟（debug_mode，无网络）    150     3324.996            22.167           0.924              NaN          NaN                      NaN
A_secure          两进程生产模式，本机回环    141    21490.553           152.415           6.354           32.148     3346.215                  880.097
每棵树耗时比 A_secure / A：6.9 倍（运行方式不同，只作同机相对量级）
已写出公开汇总： ['README.md', 'arms_dev_eval.csv', 'contrasts_dev_eval.csv', 'cost_summary.csv', 'run_summary.csv']


**读法**：同一台 2 核机器上，A_secure 每棵树约 152 秒，A 约 22 秒，约 6.9 倍（此前汇报口径约 11 倍，数据规模与运行方式不同）。两者运行方式不同（A 单进程无网络，A_secure 两进程本机回环），只作同机相对量级，不作部署耗时。通信量是更硬的约束：每棵树两方合计约 32 GiB，按 1 Gbps 链路仅传输就约 4.6 分钟/棵、150 棵约 11.5 小时，尚未计时延与通信轮数。

## 结论

1. **精度**：全程秘密分享的 SS-XGB 在同棵数下与明文集中式无可分辨差异（−0.001 [−0.004, 0.002]），相对 L1 仍有 +0.043 的增量（描述性）。
2. **成本**：每棵树耗时约为 SGB 的 6.9 倍、通信约 32 GiB；150 棵在本机约 6.35 小时，超过预注册上限，A_secure 按规则记为未完成（141 棵）。
3. **对 S0.9 的含义**：若业务方不接受 SecureBoost 类方案（含臂 A）的训练期泄漏，安全档在精度上可行，瓶颈在通信与耗时；部署前须在两地实际网络条件下复测。本步只测精度与成本，SS-XGB 训练中，每个分裂由哪一方持有、持有方的分裂阈值仍然可见，这部分披露未在本步评估。
4. 公开数据上的开发证据，不代表汇丰真实效果；技术侧自评，未经法务复核（W-001）。